In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os
from contextlib import nullcontext
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import LinearLR,SequentialLR, CosineAnnealingLR
import matplotlib.pyplot as plt
from datasets import load_dataset

In [2]:
#Encode data using GPT-2 encoder

ds = load_dataset("roneneldan/TinyStories")
enc = tiktoken.get_encoding("gpt2")

In [3]:
#Encode data

def process(example):
    ids = enc.encode_ordinary(example['text']) # encode_ordinary ignores any special tokens
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=8,
        )
    # concatenate all the ids in each dataset into one large file we can use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

In [4]:
#DATASET

# Some functions from https://github.com/karpathy/nanoGPT/blob/master/train.py with slight modifications
#block size = context window
def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


In [5]:
#LORA FINETUNING

class LoRALinear(nn.Module):
    """
    LoRA wrapper for a single nn.Linear layer.
    It adds a low-rank residual (A @ B) * scaling to the output.
    """

    def __init__(self, layer: nn.Linear, rank: int = 4, alpha: float = 1.0, device=None):
        super().__init__()
        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.in_features = layer.in_features
        self.out_features = layer.out_features

        # LoRA parameters (small)
        self.A = nn.Parameter(torch.zeros(rank, self.in_features, device=device))
        self.B = nn.Parameter(torch.zeros(self.out_features, rank, device=device))
        self.reset_parameters()

        # Freeze the original layer
        for p in self.layer.parameters():
            p.requires_grad = False

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    def forward(self, x):
        # Normal forward + LoRA residual
        result = self.layer(x)
        lora_update = (x @ self.A.T @ self.B.T) * self.scaling
        return result + lora_update
    

def inject_lora_into_transformer(model, rank=4, alpha=1.0, device=None):
    """
    Replaces each MultiHeadAttention's linear layers with LoRALinear-wrapped versions.
    """
    for name, module in model.named_modules():
        if isinstance(module, MultiHeadAttention):
            module.W_q = LoRALinear(module.W_q, rank=rank, alpha=alpha, device=device)
            module.W_k = LoRALinear(module.W_k, rank=rank, alpha=alpha, device=device)
            module.W_v = LoRALinear(module.W_v, rank=rank, alpha=alpha, device=device)
            # Optional:
            # module.W_o = LoRALinear(module.W_o, rank=rank, alpha=alpha, device=device)
            print(f"Injected LoRA into MultiHeadAttention layer: {name}")

In [6]:
# TRANSFORMER ARCHITECTURE

# --- MultiHeadAttention ---
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1, use_lora=False, lora_rank=4, lora_alpha=1.0, device = "cuda"):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        if use_lora:
            self.W_q = LoRALinear(self.W_q, rank=lora_rank, alpha=lora_alpha, device=device)
            self.W_k = LoRALinear(self.W_k, rank=lora_rank, alpha=lora_alpha, device=device)
            self.W_v = LoRALinear(self.W_v, rank=lora_rank, alpha=lora_alpha, device=device)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        return torch.matmul(attn_probs, V)

    def split_heads(self, x):
        B, T, C = x.size()
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, T, D = x.size()
        return x.transpose(1, 2).contiguous().view(B, T, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

# --- FeedForward ---
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.relu(self.fc1(x))))

# --- Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# --- Transformer Block ---
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.ln1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.ln2(x + self.dropout(ff_output))
        return x

In [7]:
@dataclass
class TransformerConfig:
    vocab_size: int
    block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.1

In [8]:
class TransformerGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.positional_encoding = PositionalEncoding(config.n_embd, config.block_size)
        self.layers = nn.ModuleList([
            TransformerBlock(config.n_embd, config.n_head, 4 * config.n_embd, config.dropout)
            for _ in range(config.n_layer)
        ])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, "Sequence length exceeds block size."

        x = self.token_embedding(idx)
        x = self.positional_encoding(x)

        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)
        logits = self.head(x)

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

In [9]:
config = TransformerConfig(
    vocab_size=50257,
    block_size=128,
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1
)
model = TransformerGPT(config)


In [10]:
""" 

from torchviz import make_dot


src_vocab_size = 100
tgt_vocab_size = 100

# Dummy input tensors (batch_size=2, seq_len=10)
src = torch.randint(1, src_vocab_size, (2, 10))
tgt = torch.randint(1, tgt_vocab_size, (2, 10))

# Forward pass
output, loss = model(src, tgt)

# Visualize computation graph for output tensor
dot = make_dot(output, params=dict(model.named_parameters()))

# Save visualization to file
dot.render("transformer_graph", format="png")


 """

' \n\nfrom torchviz import make_dot\n\n\nsrc_vocab_size = 100\ntgt_vocab_size = 100\n\n# Dummy input tensors (batch_size=2, seq_len=10)\nsrc = torch.randint(1, src_vocab_size, (2, 10))\ntgt = torch.randint(1, tgt_vocab_size, (2, 10))\n\n# Forward pass\noutput, loss = model(src, tgt)\n\n# Visualize computation graph for output tensor\ndot = make_dot(output, params=dict(model.named_parameters()))\n\n# Save visualization to file\ndot.render("transformer_graph", format="png")\n\n\n '

In [11]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

In [12]:
# Training Config

learning_rate = 1e-4 #more stable training, earlier 1e-4
max_iters = 20000 #increase from 25000
warmup_steps = 1000 #smoother initial train, earlier 100
min_lr = 5e-4 #lower rate, earlier 5e-4
eval_iters = 500 # increased from 100
batch_size = 8 # changed from 16, better gradient estimate
block_size = 128 #changed from 64, capture longer range dependencies

gradient_accumulation_steps = 32 # reduced from 50

device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
# note: float16 data type will automatically use a GradScaler

# How to use autocast https://wandb.ai/wandb_fc/tips/reports/How-To-Use-Autocast-in-PyTorch--VmlldzoyMTk4NTky
#dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

In [13]:
##PUT IN WEIGHT DECAY, CHANGED BETA2 to 0.95
optimizer =  torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9) #weight decay for regularization

scheduler_warmup = LinearLR(optimizer, total_iters = warmup_steps) #Implement linear warmup
scheduler_decay = CosineAnnealingLR(optimizer,T_max = max_iters - warmup_steps, eta_min = min_lr) #Implement lr decay
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps]) #Switching from warmup to decay

# https://stackoverflow.com/questions/72534859/is-gradscaler-necessary-with-mixed-precision-training-with-pytorch
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

/tmp/ipykernel_705313/479405836.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


In [ ]:

best_val_loss = float('inf')
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []

# Ensure model is on the correct device
model = model.to(device)

# In your training loop
for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        # Ensure estimate_loss uses the correct device
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"The current learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    # Ensure X and y are on the correct device
    X, y = get_batch("train")
    X, y = X.to(device), y.to(device)

    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    scheduler.step()
    


  0%|          | 0/20000 [00:00<?, ?it/s]

/home/tom/apps/cache/python-envs/ML/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


In [ ]:



train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()



' \n\n\ntrain_loss_list_converted = [i.cpu().detach() for i in train_loss_list]\nvalidation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]\n\nplt.plot(train_loss_list_converted, \'g\', label=\'train_loss\')\nplt.plot(validation_loss_list_converted, \'r\', label=\'validation_loss\')\nplt.xlabel("Steps - Every 100 epochs")\nplt.ylabel("Loss")\nplt.legend()\nplt.show()\n\n '

In [ ]:
model.load_state_dict(torch.load("best_model_params_orig.pt"))

<All keys matched successfully>

In [ ]:
#Load the model
model = TransformerGPT(config)  # re-create the model with same config
device =  "cuda" if torch.cuda.is_available() else "cpu"
best_model_params_path = "best_model_params_orig.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device))) # load best model states


<All keys matched successfully>

In [ ]:
sentence = "There lived a girl"
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))

There lived a girl named Tom. But her very delicate. Tom picked up her else sailedener. She waved, and waved. She screamed. She left the kitchen. She tells him out his ball. We can be careful. They say hello is busy.Lily likes to worry longer Anna paints cozy to visit her.

He gives the ball in head. He flowers, the sail on the snow. He says, "Don't know what is tight. I only not see what is friends. You said you are important."

But two dad finally nodded. He told them.

"Let's keep it if?" Lily. She painted the way to his eyes and felt. She lost that she said, Sam were wrong.

Every day in the trail to the big ladder to play with her rays.Once upon a time on, Dave a time.

They was in the park, but Tim had a big, so beautiful walk at each other. The driver said and Tim


# LORA

In [ ]:
## BEFORE LORA

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters before LoRA injection: {sum(p.numel() for p in trainable_params):,}")

Trainable parameters before LoRA injection: 49,244,928


In [ ]:
print("Total paraeters before lora: ", sum(p.numel() for p in model.parameters()))

Total paraeters before lora:  49244928


In [ ]:
## AFTER LORA

In [ ]:
inject_lora_into_transformer(model, rank=4, alpha=16, device=device)

for name, param in model.named_parameters():
    if "A" not in name and "B" not in name:
        param.requires_grad = False

# Now only LoRA parameters are trainable
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters after LoRA injection: {sum(p.numel() for p in trainable_params):,}")
print(f"Total Parameters after adding lora parameters: {sum(p.numel() for p in model.parameters())}")

Injected LoRA into MultiHeadAttention layer: layers.0.self_attn
Injected LoRA into MultiHeadAttention layer: layers.1.self_attn
Injected LoRA into MultiHeadAttention layer: layers.2.self_attn
Injected LoRA into MultiHeadAttention layer: layers.3.self_attn
Injected LoRA into MultiHeadAttention layer: layers.4.self_attn
Injected LoRA into MultiHeadAttention layer: layers.5.self_attn
Trainable parameters after LoRA injection: 55,296
Total Parameters after adding lora parameters: 49300224


In [ ]:
def merge_lora_into_base(model):
    for module in model.modules():
        if isinstance(module, LoRALinear):
            with torch.no_grad():
                W = module.layer.weight.data
                delta = (module.B @ module.A) * module.scaling
                W += delta.clone()


In [ ]:
best_val_loss = float('inf')
train_loss_list, validation_loss_list = [], []

model = model.to(device)
model.train()  # LoRA training mode (base weights frozen)

# Optimizer only over LoRA parameters
lora_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(lora_params, lr=3e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

# LoRA fine-tuning on validation dataset
for epoch in tqdm(range(max_iters)):
    X, y = get_batch("val") 
    X, y = X.to(device), y.to(device)
    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(lora_params, max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    scheduler.step()

/tmp/ipykernel_699775/4147944209.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  0%|          | 0/20000 [00:00<?, ?it/s]

/home/tom/apps/cache/python-envs/ML/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


In [ ]:
best_model_params_path = "best_lora_params.pt"

torch.save(
    {k: v for k, v in model.state_dict().items() if "A" in k or "B" in k},
    best_model_params_path
)


In [ ]:
merge_lora_into_base(model)

torch.save(model.state_dict(), "best_model_params.pt")

In [ ]:
#Load the model
#model = TransformerGPT(config)  # re-create the model with same config
device =  "cuda" if torch.cuda.is_available() else "cpu"
best_model_params_path = "best_model_params.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device))) # load best model states


<All keys matched successfully>

In [ ]:
sentence = "There lived a girl"
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))

There lived a girl named Lily: "No, Lily. She said, ran away Little we'm happy first. How gr dare you ordered a lot of noise. He was so tired!

They swam of his presence over to his toys. He asked her car away, balls. He spl Wednesday if they enjoyed the chair. 
Ben was thirsty and three that Mum. The two lesson. But Timmy said, "Don't worry, look likeworm.Sam and Tom can make lots of a lesson. Do you for a nice dress, Anna very nice more food over to spin things.

The end.Once upon a time, there was a little girl named Lily and she was playing. She loved playing outside and learned that she wanted all day up. She wanted to always watch it else.

Mia started playing in her new ocean cold space for a big bench. soon her mom some her mommy didn't touch it with blocks instead. They missed the loop animals and
